# 03. Cohort Retention and Repeat Purchase Analysis

This notebook measures monthly customer retention and 7-day/30-day repeat purchase rates. It explicitly controls for right censoring: customers without a complete observation window are excluded from the corresponding short-term rate.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.style.use('seaborn-v0_8-whitegrid')
repo_root = Path.cwd()
if repo_root.name == 'notebooks': repo_root = repo_root.parent
clean_path = repo_root / 'data' / 'processed' / 'online_retail_clean.csv.gz'
image_dir = repo_root / 'images'
output_dir = repo_root / 'data' / 'processed'
image_dir.mkdir(exist_ok=True)
df = pd.read_csv(clean_path, compression='gzip', parse_dates=['InvoiceDate'], dtype={'InvoiceNo':str,'StockCode':str,'CustomerID':str})
print(f'Loaded {len(df):,} transaction lines for {df.CustomerID.nunique():,} customers.')

## 1. Monthly cohort retention

Each customer is assigned to the month of their first purchase. `CohortIndex = 0` is the acquisition month, `1` is the next calendar month, and so on. A retained customer is one who makes at least one purchase in that activity month.

In [ ]:
customer_first_date = df.groupby('CustomerID')['InvoiceDate'].min()
df['FirstPurchaseDate'] = df['CustomerID'].map(customer_first_date)
df['CohortMonth'] = df['FirstPurchaseDate'].dt.to_period('M')
df['ActivityMonth'] = df['InvoiceDate'].dt.to_period('M')
df['CohortIndex'] = (df['ActivityMonth'].dt.year-df['CohortMonth'].dt.year)*12 + (df['ActivityMonth'].dt.month-df['CohortMonth'].dt.month)
cohort_counts = (df.groupby(['CohortMonth','CohortIndex'])['CustomerID'].nunique().unstack(fill_value=0).sort_index())
cohort_sizes = cohort_counts[0]
retention = cohort_counts.divide(cohort_sizes, axis=0)
display(cohort_counts)
display(retention)

In [ ]:
fig, ax = plt.subplots(figsize=(13,7))
matrix = retention.to_numpy(dtype=float)
masked = np.ma.masked_where(cohort_counts.to_numpy()==0, matrix)
im = ax.imshow(masked, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(retention.columns)), retention.columns)
ax.set_yticks(range(len(retention.index)), retention.index.astype(str))
ax.set_xlabel('Months since first purchase')
ax.set_ylabel('Acquisition cohort')
ax.set_title('Monthly Customer Retention by Acquisition Cohort')
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        if cohort_counts.iloc[i,j] > 0:
            color='white' if matrix[i,j] > .55 else '#111827'
            ax.text(j,i,f'{matrix[i,j]:.0%}',ha='center',va='center',fontsize=8,color=color)
fig.colorbar(im, ax=ax, label='Retention rate')
plt.tight_layout()
plt.savefig(image_dir/'06_monthly_cohort_retention.png',dpi=180,bbox_inches='tight')
plt.show()

## 2. 7-day and 30-day repeat purchase rates

For each customer, the first distinct order is the acquisition event. Repeat purchase means another distinct order after the first order and within the stated number of days. Customers acquired less than 7 or 30 days before the dataset ends are not eligible for that metric.

In [ ]:
orders = (df.groupby(['CustomerID','InvoiceNo'],as_index=False).agg(OrderDate=('InvoiceDate','min'),OrderRevenue=('Revenue','sum')).sort_values(['CustomerID','OrderDate','InvoiceNo']))
orders['OrderNumber'] = orders.groupby('CustomerID').cumcount()+1
first_orders = orders[orders.OrderNumber==1][['CustomerID','OrderDate']].rename(columns={'OrderDate':'FirstOrderDate'})
second_orders = orders[orders.OrderNumber==2][['CustomerID','OrderDate']].rename(columns={'OrderDate':'SecondOrderDate'})
repeat = first_orders.merge(second_orders,on='CustomerID',how='left')
repeat['DaysToSecondOrder'] = (repeat.SecondOrderDate-repeat.FirstOrderDate).dt.total_seconds()/86400
data_end = df.InvoiceDate.max()
repeat['CohortMonth'] = repeat.FirstOrderDate.dt.to_period('M')
for window in [7,30]:
    repeat[f'Eligible{window}D'] = repeat.FirstOrderDate <= data_end-pd.Timedelta(days=window)
    repeat[f'Repeat{window}D'] = repeat.DaysToSecondOrder.between(0,window,inclusive='right')

short_term=[]
for window in [7,30]:
    eligible=repeat[repeat[f'Eligible{window}D']]
    short_term.append({'Window':f'{window}-day','EligibleCustomers':len(eligible),'RepeatCustomers':int(eligible[f'Repeat{window}D'].sum()),'RepeatRate':eligible[f'Repeat{window}D'].mean()})
short_term=pd.DataFrame(short_term)
display(short_term)

In [ ]:
cohort_short=[]
for cohort, group in repeat.groupby('CohortMonth'):
    row={'CohortMonth':str(cohort),'Customers':len(group)}
    for window in [7,30]:
        eligible=group[group[f'Eligible{window}D']]
        row[f'Eligible{window}D']=len(eligible)
        row[f'RepeatRate{window}D']=eligible[f'Repeat{window}D'].mean() if len(eligible) else np.nan
    cohort_short.append(row)
cohort_short=pd.DataFrame(cohort_short)
display(cohort_short)
ax=cohort_short.set_index('CohortMonth')[['RepeatRate7D','RepeatRate30D']].plot(kind='bar',figsize=(12,6),color=['#60A5FA','#1D4ED8'])
ax.set_title('7-Day and 30-Day Repeat Purchase Rate by Acquisition Cohort')
ax.set_xlabel('Acquisition month')
ax.set_ylabel('Repeat purchase rate')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(image_dir/'07_short_term_repeat_purchase.png',dpi=180,bbox_inches='tight')
plt.show()

## 3. Retention summary and exports

In [ ]:
retention_export=retention.copy(); retention_export.index=retention_export.index.astype(str)
retention_export.to_csv(output_dir/'monthly_cohort_retention.csv')
cohort_short.to_csv(output_dir/'short_term_repeat_by_cohort.csv',index=False)
repeat.to_csv(output_dir/'customer_repeat_purchase.csv',index=False)
m1_weighted=cohort_counts.loc[cohort_counts.index <= pd.Period('2011-10','M'),1].sum()/cohort_sizes.loc[cohort_sizes.index <= pd.Period('2011-10','M')].sum()
rate7=short_term.loc[short_term.Window=='7-day','RepeatRate'].iloc[0]
rate30=short_term.loc[short_term.Window=='30-day','RepeatRate'].iloc[0]
print('RETENTION INSIGHT SNAPSHOT')
print(f'1. Weighted month-1 retention for mature cohorts: {m1_weighted:.1%}.')
print(f'2. 7-day repeat purchase rate: {rate7:.1%}.')
print(f'3. 30-day repeat purchase rate: {rate30:.1%}.')
print('4. Eligibility rules prevent incomplete observation windows from being counted as non-retention.')